In [9]:
import random

In [10]:
# =========================================
# Ambiente 1D (procedurale, senza global)
# =========================================

def init_env_1d(n_squares=5, dirt_prob=0.5, agent_start_pos=0, seed=None):
    """
    Inizializza l'ambiente:
    - n_squares: numero di celle
    - dirt_prob: probabilità che una cella sia sporca
    - agent_start_pos: posizione iniziale dell'agente (0 .. n_squares-1)
    - seed: opzionale, per riproducibilità
    Restituisce: (n_squares, dirt, agent_pos, performance)
    """
    if seed is not None:
        random.seed(seed)
    dirt = [random.random() < dirt_prob for _ in range(n_squares)]
    agent_pos = agent_start_pos
    performance = 0
    return n_squares, dirt, agent_pos, performance

def percept_1d(agent_pos, dirt):
    """Restituisce (posizione, è_sporco_qui)."""
    return agent_pos, dirt[agent_pos]

def step_env_1d(n_squares, dirt, agent_pos, performance, action):
    """
    Aggiorna l'ambiente in base all'azione.
    Restituisce: (dirt, agent_pos, performance) aggiornati.
    """
    performance -= 0  # costo di passo

    if action == "SUCK":
        if dirt[agent_pos]:
            dirt[agent_pos] = False
            performance += 1
    elif action == "LEFT":
        if agent_pos > 0:
            agent_pos -= 1
    elif action == "RIGHT":
        if agent_pos < n_squares - 1:
            agent_pos += 1
    elif action == "NOOP":
        pass

    return dirt, agent_pos, performance

def all_clean(dirt):
    """True se tutte le celle sono pulite."""
    return not any(dirt)

def show_env_1d(n_squares, dirt, agent_pos, performance):
    """Stampa una rappresentazione testuale dell'ambiente."""
    cells = []
    for i, d in enumerate(dirt):
        ch = "D" if d else "."
        if i == agent_pos:
            ch = "[" + ch + "]"
        cells.append(ch)
    print("".join(cells), f"(perf={performance})")


# =========================================
# Agente 1D con stato esplicito
# =========================================

def init_agent_1d(n_squares):
    """
    Stato dell'agente:
    - n_squares: numero di celle
    - direction: 1 = verso destra, -1 = verso sinistra
    """
    return {
        "n_squares": n_squares,
        "direction": 1
    }

def agent_1d_policy(agent_state, percept):
    """
    agent_state: dict con 'n_squares' e 'direction'
    percept: (pos, is_dirty)
    Restituisce (azione, nuovo_agent_state).
    L'agente:
    - SUCK se sporco
    - altrimenti pattuglia avanti e indietro tra gli estremi.
    """
    n_squares = agent_state["n_squares"]
    direction = agent_state["direction"]

    pos, is_dirty = percept

    if is_dirty:
        action = "SUCK"
    else:
        # Cambia direzione agli estremi
        if pos == 0:
            direction = 1
        elif pos == n_squares - 1:
            direction = -1
        action = "RIGHT" if direction == 1 else "LEFT"

    new_agent_state = {
        "n_squares": n_squares,
        "direction": direction
    }
    return action, new_agent_state


# =========================================
# Simulazione episodio
# =========================================

def run_episode_1d(n_squares=5, dirt_prob=0.6, steps=20,
                   agent_start_pos=0, seed=None):
    """
    Esegue un episodio di lunghezza massima 'steps'.
    Puoi impostare:
    - n_squares: numero di celle
    - dirt_prob: probabilità iniziale di sporco
    - agent_start_pos: posizione iniziale dell'agente
    - seed: per riproducibilità
    """
    n_squares, dirt, agent_pos, performance = init_env_1d(
        n_squares=n_squares,
        dirt_prob=dirt_prob,
        agent_start_pos=agent_start_pos,
        seed=seed
    )
    agent_state = init_agent_1d(n_squares)

    for t in range(steps):
        print(f"t={t:02d}:", end=" ")
        show_env_1d(n_squares, dirt, agent_pos, performance)

        percept = percept_1d(agent_pos, dirt)
        action, agent_state = agent_1d_policy(agent_state, percept)

        dirt, agent_pos, performance = step_env_1d(
            n_squares, dirt, agent_pos, performance, action
        )

        if all_clean(dirt):
            print(f"t={t+1:02d}:", end=" ")
            show_env_1d(n_squares, dirt, agent_pos, performance)
            print("Tutto pulito, episodio terminato.")
            break


In [11]:
if __name__ == "__main__":
    # agente parte dalla cella 3 in un ambiente di 7 celle
    run_episode_1d(
        n_squares=20,
        dirt_prob=0.4,
        steps=60,
        agent_start_pos=3,
        seed=42
    )


t=00: .DD[D]...D.DD.DD..D..D (perf=0)
t=01: .DD[.]...D.DD.DD..D..D (perf=1)
t=02: .DD.[.]..D.DD.DD..D..D (perf=1)
t=03: .DD..[.].D.DD.DD..D..D (perf=1)
t=04: .DD...[.]D.DD.DD..D..D (perf=1)
t=05: .DD....[D].DD.DD..D..D (perf=1)
t=06: .DD....[.].DD.DD..D..D (perf=2)
t=07: .DD.....[.]DD.DD..D..D (perf=2)
t=08: .DD......[D]D.DD..D..D (perf=2)
t=09: .DD......[.]D.DD..D..D (perf=3)
t=10: .DD.......[D].DD..D..D (perf=3)
t=11: .DD.......[.].DD..D..D (perf=4)
t=12: .DD........[.]DD..D..D (perf=4)
t=13: .DD.........[D]D..D..D (perf=4)
t=14: .DD.........[.]D..D..D (perf=5)
t=15: .DD..........[D]..D..D (perf=5)
t=16: .DD..........[.]..D..D (perf=6)
t=17: .DD...........[.].D..D (perf=6)
t=18: .DD............[.]D..D (perf=6)
t=19: .DD.............[D]..D (perf=6)
t=20: .DD.............[.]..D (perf=7)
t=21: .DD..............[.].D (perf=7)
t=22: .DD...............[.]D (perf=7)
t=23: .DD................[D] (perf=7)
t=24: .DD................[.] (perf=8)
t=25: .DD...............[.]. (perf=8)
t=26: .DD...